In [16]:
import scanpy as sc
import umap
import numpy as np
import matplotlib.pyplot as plt

from sklearn.manifold import trustworthiness, TSNE
from metrics import pairwise_distance_correlation, knn_accuracy, knn_recall
from sklearn.metrics import silhouette_score

from IsUMap import IsUMap

import time

**Загрузка данных**

In [17]:
adata = sc.datasets.pbmc3k()

In [18]:
print(adata)
print(adata.X.shape)

AnnData object with n_obs × n_vars = 2700 × 32738
    var: 'gene_ids'
(2700, 32738)


In [19]:
adata.obs

""
index
AAACATACAACCAC-1
AAACATTGAGCTAC-1
AAACATTGATCAGC-1
AAACCGTGCTTCCG-1
AAACCGTGTATGCG-1
...
TTTCGAACTCTCAT-1
TTTCTACTGAGGCA-1
TTTCTACTTCCTCG-1


In [20]:
adata.var

,gene_ids
index,
MIR1302-10,ENSG00000243485
FAM138A,ENSG00000237613
OR4F5,ENSG00000186092
RP11-34P13.7,ENSG00000238009
RP11-34P13.8,ENSG00000239945
...,...
AC145205.1,ENSG00000215635
BAGE5,ENSG00000268590
CU459201.1,ENSG00000251180


In [21]:
sc.tl.pca(adata, n_comps=50)

adata.obsm['X_pca']

array([[-13.06409   ,  39.60459   ,   1.6050348 , ...,  -4.023971  ,
          2.2566109 ,  -7.410456  ],
       [ 48.334526  , 197.30684   , -10.306313  , ..., -14.029236  ,
          7.4008164 ,  -7.208507  ],
       [  7.2713213 ,  79.56265   ,  87.01481   , ...,  -7.437106  ,
          5.232172  ,   4.9185343 ],
       ...,
       [-26.113253  , -24.414034  , -23.46927   , ...,  -0.8729378 ,
         -3.9896078 ,   4.107026  ],
       [-63.81711   , -47.97409   ,  29.616089  , ...,  -0.85719013,
         -1.8567727 ,   0.49989653],
       [-32.456596  ,   7.895515  , -14.203711  , ...,   1.3257372 ,
          5.727844  ,  -5.266928  ]], shape=(2700, 50), dtype=float32)

In [22]:
X = adata.obsm['X_pca'][:, :50].astype(np.float32)

In [23]:
n_neighbors_list = [10, 30, 50, 100]
n_runs = 5
seeds = [42 + i for i in range(n_runs)]


**Применение UMAP**

In [24]:
X_umap_list = []
umap_times = []
for el in n_neighbors_list:
    X_umap_runs = []
    umap_time_runs = []
    for seed in seeds:
        start = time.perf_counter()
        reducer = umap.UMAP(
            n_neighbors=el,
            metric='euclidean',
            random_state=seed,
        )
        X_umap = reducer.fit_transform(X)
        end = time.perf_counter()

        umap_time_runs.append(end - start)
        X_umap_runs.append(X_umap)

    umap_times.append(umap_time_runs)
    X_umap_list.append(X_umap_runs)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no 

**Применение IsUMap**

In [25]:
X_isumap_list = []
isumap_times = []
for el in n_neighbors_list:
    X_isumap_runs = []
    isumap_time_runs = []
    for seed in seeds:
        start = time.perf_counter()
        isumap = IsUMap(
            n_neighbors=el,
            n_components=2,
            metric='euclidean',
            mode='um',
            use_rho=True,
            random_state=seed,
        )
        X_isumap = isumap.fit_transform(X)
        end = time.perf_counter()

        isumap_time_runs.append(end - start)
        X_isumap_runs.append(X_isumap)

    isumap_times.append(isumap_time_runs)
    X_isumap_list.append(X_isumap_runs)


**Сравнение UMAP и IsUMap**

In [26]:
# Time
for i in range(len(n_neighbors_list)):
    print(f'n_neighbors = {n_neighbors_list[i]}')
    print(f'UMAP time: {np.mean(umap_times[i]):.6f} +- {np.std(umap_times[i]):.6f}')
    print(f'IsUMap time: {np.mean(isumap_times[i]):.6f} +- {np.std(isumap_times[i]):.6f}')
    print()


n_neighbors = 10
UMAP time: 5.249019 +- 0.051321
IsUMap time: 4.535453 +- 0.026534

n_neighbors = 30
UMAP time: 13.364358 +- 0.021397
IsUMap time: 11.227392 +- 0.168951

n_neighbors = 50
UMAP time: 21.298474 +- 0.121646
IsUMap time: 16.810477 +- 0.088168

n_neighbors = 100
UMAP time: 39.743379 +- 0.062376
IsUMap time: 31.753139 +- 0.042891



In [27]:
# Trustworthiness
for i in range(len(n_neighbors_list)):
    print(f'n_neighbors = {n_neighbors_list[i]}')
    tw_umap_runs = [trustworthiness(X, emb, n_neighbors=10, metric='euclidean') for emb in X_umap_list[i]]
    tw_isumap_runs = [trustworthiness(X, emb, n_neighbors=10, metric='euclidean') for emb in X_isumap_list[i]]

    print(f'UMAP trustworthiness: {np.mean(tw_umap_runs):.6f} +- {np.std(tw_umap_runs):.6f}')
    print(f'IsUMap trustworthiness: {np.mean(tw_isumap_runs):.6f} +- {np.std(tw_isumap_runs):.6f}')
    print()


n_neighbors = 10
UMAP trustworthiness: 0.950013 +- 0.000071
IsUMap trustworthiness: 0.890384 +- 0.000000

n_neighbors = 30
UMAP trustworthiness: 0.944273 +- 0.000316
IsUMap trustworthiness: 0.892753 +- 0.000000

n_neighbors = 50
UMAP trustworthiness: 0.940355 +- 0.000346
IsUMap trustworthiness: 0.894928 +- 0.000000

n_neighbors = 100
UMAP trustworthiness: 0.928985 +- 0.000358
IsUMap trustworthiness: 0.896892 +- 0.000000



In [28]:
# pairwise_distance_correlation
for i in range(len(n_neighbors_list)):
    print(f'n_neighbors = {n_neighbors_list[i]}')
    sp_corr_umap_runs = [pairwise_distance_correlation(X, emb, method='spearman')[0] for emb in X_umap_list[i]]
    sp_corr_isumap_runs = [pairwise_distance_correlation(X, emb, method='spearman')[0] for emb in X_isumap_list[i]]

    print(f'UMAP spearman corr: {np.mean(sp_corr_umap_runs):.6f} +- {np.std(sp_corr_umap_runs):.6f}')
    print(f'IsUMap spearman corr: {np.mean(sp_corr_isumap_runs):.6f} +- {np.std(sp_corr_isumap_runs):.6f}')
    print()


n_neighbors = 10
UMAP spearman corr: 0.699841 +- 0.014558
IsUMap spearman corr: 0.660066 +- 0.000000

n_neighbors = 30
UMAP spearman corr: 0.706539 +- 0.003041
IsUMap spearman corr: 0.664961 +- 0.000000

n_neighbors = 50
UMAP spearman corr: 0.672567 +- 0.015537
IsUMap spearman corr: 0.674704 +- 0.000000

n_neighbors = 100
UMAP spearman corr: 0.716020 +- 0.007228
IsUMap spearman corr: 0.685353 +- 0.000000



In [29]:
# knn-recall
for i in range(len(n_neighbors_list)):
    print(f'n_neighbors = {n_neighbors_list[i]}')
    knn_recall_umap_runs = [knn_recall(X, emb) for emb in X_umap_list[i]]
    knn_recall_isumap_runs = [knn_recall(X, emb) for emb in X_isumap_list[i]]

    print(f'UMAP kNN recall: {np.mean(knn_recall_umap_runs):.6f} +- {np.std(knn_recall_umap_runs):.6f}')
    print(f'IsUMap kNN recall: {np.mean(knn_recall_isumap_runs):.6f} +- {np.std(knn_recall_isumap_runs):.6f}')
    print()


n_neighbors = 10
UMAP kNN recall: 0.254030 +- 0.001810
IsUMap kNN recall: 0.198000 +- 0.000000

n_neighbors = 30
UMAP kNN recall: 0.193919 +- 0.000752
IsUMap kNN recall: 0.200000 +- 0.000000

n_neighbors = 50
UMAP kNN recall: 0.174311 +- 0.001631
IsUMap kNN recall: 0.200185 +- 0.000000

n_neighbors = 100
UMAP kNN recall: 0.138096 +- 0.001224
IsUMap kNN recall: 0.199074 +- 0.000000



**Применение t-SNE и метрики**

In [30]:
X_tsne_list = []
tsne_times = []
for el in n_neighbors_list:
    X_tsne_runs = []
    tsne_time_runs = []
    for seed in seeds:
        start = time.perf_counter()
        tsne = TSNE(
            n_components=2,
            perplexity=el,
            metric='euclidean',
            init='pca',
            learning_rate='auto',
            random_state=seed,
        )
        X_tsne = tsne.fit_transform(X)
        end = time.perf_counter()

        tsne_time_runs.append(end - start)
        X_tsne_runs.append(X_tsne)

    tsne_times.append(tsne_time_runs)
    X_tsne_list.append(X_tsne_runs)


In [31]:
for i in range(len(n_neighbors_list)):
    print(f'perplexity = {n_neighbors_list[i]}')
    print(f't-SNE time: {np.mean(tsne_times[i]):.6f} +- {np.std(tsne_times[i]):.6f}')

    tw_tsne_runs = [trustworthiness(X, emb, n_neighbors=10, metric='euclidean') for emb in X_tsne_list[i]]
    print(f't-SNE trustworthiness: {np.mean(tw_tsne_runs):.6f} +- {np.std(tw_tsne_runs):.6f}')

    sp_corr_tsne_runs = [pairwise_distance_correlation(X, emb, method='spearman')[0] for emb in X_tsne_list[i]]
    print(f't-SNE spearman corr: {np.mean(sp_corr_tsne_runs):.6f} +- {np.std(sp_corr_tsne_runs):.6f}')

    knn_recall_tsne_runs = [knn_recall(X, emb) for emb in X_tsne_list[i]]
    print(f't-SNE kNN recall: {np.mean(knn_recall_tsne_runs):.6f} +- {np.std(knn_recall_tsne_runs):.6f}')
    print()


perplexity = 10
t-SNE time: 2.920030 +- 0.037877
t-SNE trustworthiness: 0.965302 +- 0.000120
t-SNE spearman corr: 0.627029 +- 0.001361
t-SNE kNN recall: 0.369030 +- 0.000966

perplexity = 30
t-SNE time: 3.614585 +- 0.118784
t-SNE trustworthiness: 0.968595 +- 0.000147
t-SNE spearman corr: 0.705955 +- 0.000412
t-SNE kNN recall: 0.376378 +- 0.000269

perplexity = 50
t-SNE time: 3.976307 +- 0.036715
t-SNE trustworthiness: 0.966836 +- 0.000083
t-SNE spearman corr: 0.746334 +- 0.000301
t-SNE kNN recall: 0.361407 +- 0.000509

perplexity = 100
t-SNE time: 4.355952 +- 0.067131
t-SNE trustworthiness: 0.964325 +- 0.000052
t-SNE spearman corr: 0.760080 +- 0.000108
t-SNE kNN recall: 0.317185 +- 0.000099

